# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll walk through loading metadata and records, reviewing the schema, extracting the main data, and preparing it for analysis—all by referencing entities (record sets, fields, columns) via their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View high-level metadata attributes
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Date Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** Every entity (record set, field, column) is referenced by its `@id`. This ensures consistency and reproducibility.

In [ ]:
# List all record sets available in the dataset
print("Available record sets (by @id):")
record_sets = [r['@id'] for r in dataset.metadata.recordSets]
for i, rec_id in enumerate(record_sets):
    print(f"{i+1}. {rec_id}")

# For each record set, show its fields (columns) and their @id
for record_set in dataset.metadata.recordSets:
    print(f"\nRecord Set: {record_set['@id']}")
    print("Description:", record_set.get('description', 'No description'))
    # List the columns (@id and name/label if available)
    if 'fields' in record_set:
        print("  Fields (columns):")
        for field in record_set['fields']:
            print(f"    - {field['@id']}", end='')
            label = field.get('name') or field.get('label')
            if label:
                print(f"  (name: {label})")
            else:
                print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the record set and field `@id`s found above. Here, we'll extract the main protein record set and preview its data.

In [ ]:
# Choose the primary record set @id from overview (replace with the exact @id as needed)
primary_record_set_id = dataset.metadata.recordSets[0]['@id']  # Example: 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-records'
# Optionally extend to all record sets if desired:
all_record_set_ids = [r['@id'] for r in dataset.metadata.recordSets]

dfs = {}
for rec_id in all_record_set_ids:
    print(f"Loading records for record set: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    dfs[rec_id] = pd.DataFrame(records)

df = dfs[primary_record_set_id]
print(f"Columns for {primary_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping data by key attributes. Entities are always referenced by their `@id` (see overview above for your dataset's options).

Below, we'll filter by a numeric field (e.g., molecular weight or abundance), normalize it, and summarize by a grouping field (if appropriate).

In [ ]:
# Inspect the available numeric fields in columns
print("Numeric columns candidate list:")
print(df.select_dtypes(include=['int64', 'float64']).columns.tolist())

# Select one numeric field by its @id (e.g., MW, abundance; adjust as per your schema)
numeric_field_id = df.select_dtypes(include=['int64', 'float64']).columns[0]  # Use the first numeric field
print(f"Selected numeric field: {numeric_field_id}")

# Filtering: show only rows with value > threshold
threshold = df[numeric_field_id].quantile(0.9)
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by another field if available
possible_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
We'll visualize the distribution of the selected numeric field and (if available) compare means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If a grouping field is available, plot means
if group_field:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook guided you through loading and exploring the FAIR² dataset using the `mlcroissant` library, always referencing entities by their `@id`. Adjust field and record set identifiers as needed using your dataset's schema overview for deeper or alternative analyses.